# Building a Doubly Controlled Modular Multiplier
To build  Shor's Factorizaiton Algorithm, we need the following unitary operator:
$$
U |c\rangle_1 |y\rangle_n  = \begin{cases} 
|c\rangle |ay \pmod N\rangle  & \text{if } c=1 \text{ and } y < N \\
|c\rangle |y\rangle & \text{otherwise}
\end{cases}
$$
for given $a,N\in \mathbb{Z}_+$ such that $a<N$, $gcd(a,N)=1$ and $n=\lceil \log_{2}(N) \rceil$.


In [40]:
import math
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.synthesis.qft import synth_qft_full

## Fourier Domain Phase Additions
We define the following phase addition operations. In the Fourier domain, adding a constant $a$ corresponds to applying a state dependent phase shift. The unitary transformation performs the following mapping:

$$
|x\rangle \rightarrow \exp\left( \frac{2\pi i a x}{2^{n+1}} \right) |x\rangle
$$
Because our modular operations will eventually be conditioned on multiple control qubits, we define both single-controlled and doubly-controlled versions of these phase gates.

In [41]:
def apply_phase_add(circuit, val, targets, sign=1):
    for j in range(len(targets)):
        phase = sign * 2 * np.pi * val * (2**j) / (2**len(targets))
        circuit.p(phase, targets[j])
        
def apply_ctrl_phase_add(circuit, val, control, targets, sign=1):
    for j in range(len(targets)):
        phase = sign * 2 * np.pi * val * (2**j) / (2**len(targets))
        circuit.cp(phase, control, targets[j])

def apply_doubly_ctrl_phase(circuit, val, ctrl1, ctrl2, targets, sign=1):
    for j in range(len(targets)):
        phase = sign * 2 * np.pi * val * (2**j) / (2**len(targets))
        circuit.mcp(phase, [ctrl1, ctrl2], targets[j])

## The Core Multiplier Block

Our goal is to construct a unitary gate that performs controlled modular multiplication into an accumulator register $x$, applying the transformation:
$$
|y\rangle |x\rangle \rightarrow |y\rangle |(x + a \cdot y) \pmod N\rangle
$$

### Step A: Draper's QFT Addition
We start with **Draper's QFT addition algorithm**. On a standard $n$-qubit register, Draper's algorithm applies the Quantum Fourier Transform over $2^{n}$, rotates the phases by a constant $a$, and applies the Inverse QFT to output:
$$
|x\rangle \rightarrow |(x + a) \pmod{2^n}\rangle
$$

### Step B: Upgrading to Modulo $N$
Because we defined $n = \lceil \log_2(N) \rceil$, the sum $x+a$ can exceed $N$. To figure out $(x+a) \pmod N$, we can subtract $N$ from the sum and simply check if the result is negative. 

In order to do this, we add an **extra qubit**, expanding our $x$ register to $n+1$ qubits, and perform our Fourier transforms over $2^{n+1}$. The most significant bit of this expanded register (The rightmost bit) will be $1$ **if and only if** there is an underflow (meaning $x+a < N$). 


We copy this MSB to a separate ancilla qubit using a CNOT gate. If the ancilla is $1$, we know an underflow occurred, and we conditionally add $N$ back to restore the correct modulo target. 

### Step C: Scaling to Multiplication
To construct a multiplier from our adder, we observe the following relation:

$$
ay \pmod N = \sum y_i (a 2^i \pmod N) \pmod N
$$

This demonstrates that modular multiplication is simply a series of modular additions of the shifted constants $(a 2^i \pmod N)$, where each addition is controlled by the corresponding $y_i$ qubit.

### Step D: Restoring the Ancilla Qubit

Ancilla qubit needs to be reset to $|0\rangle$ at each step for the future use. The following fact implies that it suffices to subtract $a_i = a 2^i \pmod N$), and check the sign of the resulting number, which is essentially the same with the Step B. 

$$
(x + a_i) \pmod N \ge a_i \iff x + a_i < N
$$
.

In [42]:
def append_ctrl_mod_mult(qc, a_val, N, c_qubit, y_reg, x_reg, ancilla_qubit):
    """The core doubly-controlled modular multiplier block."""
    n = len(y_reg)
    x_len = len(x_reg)

    qft_circ = synth_qft_full(x_len, do_swaps=True)
    iqft_circ = qft_circ.inverse()

    for i in range(n):
        a_shifted = (a_val * (2**i)) % N 
        if a_shifted == 0: continue
            
        qc.compose(qft_circ, qubits=x_reg, inplace=True)
        apply_doubly_ctrl_phase(qc, a_shifted, c_qubit, y_reg[i], x_reg, sign=1)
        apply_phase_add(qc, N, x_reg, sign=-1)
        qc.compose(iqft_circ, qubits=x_reg, inplace=True)
        
        qc.cx(x_reg[-1], ancilla_qubit)
        
        qc.compose(qft_circ, qubits=x_reg, inplace=True)
        apply_ctrl_phase_add(qc, N, ancilla_qubit, x_reg, sign=1)
        apply_doubly_ctrl_phase(qc, a_shifted, c_qubit, y_reg[i], x_reg, sign=-1)
        qc.compose(iqft_circ, qubits=x_reg, inplace=True)
        
        qc.x(x_reg[-1])
        qc.cx(x_reg[-1], ancilla_qubit)
        qc.x(x_reg[-1])
        
        qc.compose(qft_circ, qubits=x_reg, inplace=True)
        apply_doubly_ctrl_phase(qc, a_shifted, c_qubit, y_reg[i], x_reg, sign=1)
        qc.compose(iqft_circ, qubits=x_reg, inplace=True)

##  The Controlled Multiplier

To finalize our modular multiplier, we must add  control conditions and convert the operation to run entirely on the $y$ register.

### Step A:  Control Qubits
We want the gate to execute only if two conditions are met:
1. The control qubit $c$ is $|1\rangle$.
2. Our input register $y$ is strictly less than $N$. 

To verify $y < N$, we temporarily use the empty $x$ register to compute $y - N$. Just like in our addition block, we check the most significant bit (MSB). If the MSB is $1$, we know $y - N < 0$, which guarantees $y < N$. We copy this MSB to a flag qubit, cleanly uncompute the $x$ register back to $|0\rangle$, and use a Toffoli (CCX) gate to combine our external control $c$ and this flag qubit into a single **effective control** qubit.

### Step B: Constructing the Final Unitary Mapping
Our core gate currently takes inputs $|y\rangle |x\rangle$ and outputs $|y\rangle |(x + ay) \pmod N\rangle$. Assuming the $x$ register is initialized to $|0\rangle$, this yields $|y\rangle |ay \pmod N\rangle$. 

To turn this into a gate that maps $|y\rangle \rightarrow |ay \pmod N\rangle$ while leaving $x$ empty, we use a three-part trick:

1. **Forward Multiplier:** We apply our core multiplier block to generate the state $|y\rangle |ay \pmod N\rangle$.
2. **Register Swap:** We swap $y$ and $x$ registers. Our state is now $|ay \pmod N\rangle |y\rangle$. 
3. **Inverse Multiplication:** To clear the $x$ register (which now holds the original $y$) back to $|0\rangle$, we run the core multiplier block with $N-a^{-1}$. The existence of the inverse is guaranteed by $gcd(a,N)=1$.


By clearing the $x$ register, our final transformation becomes the desired unitary transformation:

$$
|c\rangle |y\rangle |0\rangle \rightarrow |c\rangle |ay \pmod N\rangle |0\rangle
$$

In [45]:
def build_conditional_multiplier(a: int, N: int):
    """Builds the full gate: |c>|y>|0> -> |c>|a*y mod N>|0>"""
    n = math.ceil(math.log2(N))
    x_len = n + 1 
    
    c_reg = QuantumRegister(1, name='c')
    y_reg = QuantumRegister(n, name='y')
    x_reg = QuantumRegister(x_len, name='x')
    ancilla_qubit = QuantumRegister(1, name='ancilla_qubit')
    flag = QuantumRegister(1, name='y_less_than_N')
    c_eff = QuantumRegister(1, name='c_eff')
    
    qc = QuantumCircuit(c_reg, y_reg, x_reg, ancilla_qubit, flag, c_eff, name=f"Cond_U_{a}")
    
    qft_circ = synth_qft_full(x_len, do_swaps=True)
    iqft_circ = qft_circ.inverse()
    
    def compute_less_than_N(circuit):
        circuit.compose(qft_circ, qubits=x_reg, inplace=True)
        for i in range(n):
            apply_ctrl_phase_add(circuit, 2**i, y_reg[i], x_reg, sign=1)
        apply_phase_add(circuit, N, x_reg, sign=-1)
        circuit.compose(iqft_circ, qubits=x_reg, inplace=True)
        
        circuit.cx(x_reg[-1], flag[0])
        
        circuit.compose(qft_circ, qubits=x_reg, inplace=True)
        apply_phase_add(circuit, N, x_reg, sign=1)
        for i in range(n):
            apply_ctrl_phase_add(circuit, 2**i, y_reg[i], x_reg, sign=-1)
        circuit.compose(iqft_circ, qubits=x_reg, inplace=True)

    compute_less_than_N(qc)
    qc.ccx(c_reg[0], flag[0], c_eff[0])
    
    append_ctrl_mod_mult(qc, a, N, c_eff[0], y_reg, x_reg, ancilla_qubit[0])
    
    for i in range(n):
        qc.cswap(c_eff[0], y_reg[i], x_reg[i])
        
    a_inv = pow(a, -1, N) 
    a_inv_neg = (N - a_inv) % N 
    append_ctrl_mod_mult(qc, a_inv_neg, N, c_eff[0], y_reg, x_reg, ancilla_qubit[0])
    
    #qc.ccx(c_reg[0], flag[0], c_eff[0])
    #compute_less_than_N(qc)
    
    return qc

## Test
We set $N=15$ and $a=4$. We will initialize our state to $y=6$ (binary `0110`). Because $6 <15$, the expected output is 9 (binary `1001`). Simulator runs the same circuit 1000 times and counts the each measured outcome.

In [ ]:
a = 4
N = 15
n = math.ceil(math.log2(N))
x_len = n + 1

total_qubits = 1 + n + x_len + 3
test_qc = QuantumCircuit(total_qubits, n)

# Set c = 1
test_qc.x(0) 

# Set y = 6 (Flip the '2' and '4' bit indices)
test_qc.x(2) 
test_qc.x(3) 
 

test_qc.barrier()

cond_circuit = build_conditional_multiplier(a, N)
test_qc.compose(cond_circuit, inplace=True)

test_qc.measure(range(1, n+1), range(n))

simulator = AerSimulator()
compiled_circuit = transpile(test_qc, simulator)
result = simulator.run(compiled_circuit, shots=1000).result()

print(f"Test: c=1, y=6, a=4, N=15")
print(f"Results: {result.get_counts()}")

Test: c=1, y=6, a=4, N=15
Results: {'1001': 1000}
